# Stage 7: Model Optimization & Final Model Selection

This notebook takes the Stage 6 baselines and improves them.

**What it does**
1. Loads the processed data and encodes the target labels.
2. Investigates **feature selection** (all features vs. top 100) using a leakage-safe pipeline.
3. Loads the **Stage 6 baseline results** for comparison.
4. Tunes **Random Forest** and **XGBoost** with `RandomizedSearchCV`.
5. Compares baseline vs. tuned models.
6. Selects the **final model using CV Macro F1 only**.
7. Evaluates the final model once on the **hold-out test set**, then saves the model and result files.

**Prerequisites**
- Processed CSVs in `data/processed/` (`X_train_final.csv`, `X_test_final.csv`, `y_train_final.csv`, `y_test_final.csv`).
- `stage6_model_comparison_results.csv` produced by the Stage 6 notebook.

## 1. Imports

Standard tools (`time`, `Path`, `joblib`, `numpy`, `pandas`) plus scikit-learn components for modelling, feature selection, pipelines, tuning and metrics.

XGBoost is imported inside a `try/except`, so the notebook still runs if it is not installed. In that case, XGBoost tuning is skipped.

In [ ]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    StratifiedKFold,
    RandomizedSearchCV,
    cross_validate,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

try:
    from xgboost import XGBClassifier

    XGB_AVAILABLE = True

except ImportError:
    XGB_AVAILABLE = False

## 2. Configuration

- `DATA_DIR` is the location of the processed data.
- `RANDOM_STATE` keeps every random step reproducible.

In [ ]:
# ============================================================
# STAGE 7:
# MODEL OPTIMIZATION & FINAL MODEL SELECTION
# ============================================================

DATA_DIR = Path("data/processed")
RANDOM_STATE = 42

## 3. The `main()` function

The whole stage is defined in one function, and it is kept exactly as written so the code is unchanged. Its parts, in order:

**[1/7] Load data and encode labels:** loads train/test data and encodes labels with `LabelEncoder` (fitted on training labels only). Sets up Stratified 5-Fold CV.

**[2/7] Feature selection investigation:** compares a Random Forest on **all features** with one on the **top 100 features**. `SelectFromModel` sits *inside* the `Pipeline`, so features are selected on each training fold only. This prevents feature-selection leakage.

**[3/7] Load Stage 6 baseline results:** reads `stage6_model_comparison_results.csv` and raises an error if Stage 6 has not been run.

**[4/7] Hyperparameter tuning:** `RandomizedSearchCV` (10 iterations, macro F1 scoring) for Random Forest and, if available, XGBoost.

**[5/7] Baseline vs. tuned comparison:** shows CV F1 improvement over the Stage 6 baselines and saves it to CSV.

**[6/7] Final model selection:** ranks tuned models by CV Macro F1 and picks the best. The test set plays no part in this decision.

**[7/7] Final test evaluation:** scores the chosen model on the hold-out test set with a classification report and confusion matrix. Saves the model (`final_optimized_model.joblib`), the label encoder and several result CSVs.

The function ends with a printed **Stage 8 / viva technical justification**.

In [ ]:
def main():

    print("=" * 80)
    print("STAGE 7: MODEL OPTIMIZATION & FINAL MODEL SELECTION")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Load processed datasets
    # --------------------------------------------------------

    print("\n[1/7] Loading processed datasets...")

    X_train_final = pd.read_csv(
        DATA_DIR / "X_train_final.csv"
    )

    X_test_final = pd.read_csv(
        DATA_DIR / "X_test_final.csv"
    )

    y_train_raw = pd.read_csv(
        DATA_DIR / "y_train_final.csv"
    ).values.ravel()

    y_test_raw = pd.read_csv(
        DATA_DIR / "y_test_final.csv"
    ).values.ravel()

    # --------------------------------------------------------
    # Encode target labels
    # --------------------------------------------------------
    # XGBoost requires numeric labels for multiclass
    # classification.
    # --------------------------------------------------------

    label_encoder = LabelEncoder()

    y_train = label_encoder.fit_transform(
        y_train_raw
    )

    y_test = label_encoder.transform(
        y_test_raw
    )

    class_names = [
        str(cls)
        for cls in label_encoder.classes_
    ]

    print(
        f"  Training samples : "
        f"{X_train_final.shape[0]:,}"
    )

    print(
        f"  Test samples     : "
        f"{X_test_final.shape[0]:,}"
    )

    print(
        f"  Features         : "
        f"{X_train_final.shape[1]:,}"
    )

    print(
        "  Classes          : "
        f"{dict(enumerate(class_names))}"
    )

    # --------------------------------------------------------
    # Validation strategy
    # --------------------------------------------------------

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    # ========================================================
    # 2. FEATURE SELECTION INVESTIGATION
    # ========================================================

    print(
        "\n[2/7] Investigating Feature Selection "
        "Using Cross-Validation..."
    )

    print(
        "  Comparing all features against the "
        "top 100 selected features."
    )

    # --------------------------------------------------------
    # All-feature Random Forest
    # --------------------------------------------------------

    rf_all_pipeline = Pipeline(
        steps=[
            (
                "model",
                RandomForestClassifier(
                    n_estimators=100,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            )
        ]
    )

    all_feature_scores = cross_validate(
        rf_all_pipeline,
        X_train_final,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=1,
        return_train_score=False,
    )

    all_features_cv_f1 = (
        all_feature_scores["test_score"].mean()
    )

    all_features_cv_std = (
        all_feature_scores["test_score"].std()
    )

    # --------------------------------------------------------
    # Top-100 feature pipeline
    # --------------------------------------------------------
    #
    # IMPORTANT:
    # Feature selection is INSIDE the pipeline.
    #
    # Therefore, during every CV fold:
    #
    # Training fold
    #       ↓
    # Feature selection
    #       ↓
    # Model training
    #       ↓
    # Validation fold
    #
    # This prevents feature-selection leakage.
    # --------------------------------------------------------

    top100_pipeline = Pipeline(
        steps=[
            (
                "feature_selection",
                SelectFromModel(
                    estimator=RandomForestClassifier(
                        n_estimators=50,
                        random_state=RANDOM_STATE,
                        n_jobs=-1,
                    ),
                    threshold=-np.inf,
                    max_features=100,
                ),
            ),

            (
                "model",
                RandomForestClassifier(
                    n_estimators=100,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]
    )

    top100_scores = cross_validate(
        top100_pipeline,
        X_train_final,
        y_train,
        cv=cv,
        scoring="f1_macro",
        n_jobs=1,
        return_train_score=False,
    )

    top100_cv_f1 = (
        top100_scores["test_score"].mean()
    )

    top100_cv_std = (
        top100_scores["test_score"].std()
    )

    feature_results = pd.DataFrame(
        [
            {
                "Feature Set": "All Features",
                "Feature Count": X_train_final.shape[1],
                "CV Macro F1": all_features_cv_f1,
                "CV F1 Std": all_features_cv_std,
            },
            {
                "Feature Set": "Top 100 Features",
                "Feature Count": 100,
                "CV Macro F1": top100_cv_f1,
                "CV F1 Std": top100_cv_std,
            },
        ]
    )

    print(
        "\nFeature Selection Comparison:"
    )

    print(
        feature_results.to_string(
            index=False,
            float_format=lambda value: f"{value:.4f}",
        )
    )

    feature_results.to_csv(
        "stage7_feature_selection_results.csv",
        index=False,
    )

    if all_features_cv_f1 >= top100_cv_f1:

        selected_feature_strategy = (
            "All Features"
        )

        print(
            "\nObservation: The all-feature representation "
            "achieved equal or higher CV Macro F1."
        )

    else:

        selected_feature_strategy = (
            "Top 100 Features"
        )

        print(
            "\nObservation: The top-100 feature representation "
            "achieved higher CV Macro F1."
        )

    # ========================================================
    # 3. LOAD STAGE 6 BASELINE RESULTS
    # ========================================================

    print(
        "\n[3/7] Loading Stage 6 baseline results..."
    )

    baseline_path = Path(
        "stage6_model_comparison_results.csv"
    )

    if not baseline_path.exists():

        raise FileNotFoundError(
            "stage6_model_comparison_results.csv "
            "was not found. Run Stage 6 first."
        )

    stage6_df = pd.read_csv(
        baseline_path
    )

    print(
        "  Stage 6 baseline results loaded."
    )

    # Helper function for finding baseline row
    def get_baseline(model_name):

        row = stage6_df[
            stage6_df["Model"] == model_name
        ]

        if row.empty:
            return None

        return row.iloc[0]

    # ========================================================
    # 4. HYPERPARAMETER TUNING
    # ========================================================

    print(
        "\n[4/7] Hyperparameter Optimization "
        "Using RandomizedSearchCV..."
    )

    tuned_searches = {}

    # --------------------------------------------------------
    # 4A. Random Forest
    # --------------------------------------------------------

    print(
        "\n[A] Tuning Random Forest..."
    )

    rf_param_dist = {

        "n_estimators": [
            100,
            200,
            300,
        ],

        "max_depth": [
            None,
            15,
            25,
            35,
        ],

        "min_samples_split": [
            2,
            5,
            10,
        ],

        "min_samples_leaf": [
            1,
            2,
            4,
        ],

        "max_features": [
            "sqrt",
            "log2",
        ],
    }

    rf_base = RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    rf_search = RandomizedSearchCV(
        estimator=rf_base,
        param_distributions=rf_param_dist,
        n_iter=10,
        cv=cv,
        scoring="f1_macro",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
        refit=True,
    )

    start_time = time.time()

    rf_search.fit(
        X_train_final,
        y_train,
    )

    rf_time = time.time() - start_time

    tuned_searches[
        "Random Forest"
    ] = rf_search

    print(
        f"  Completed in {rf_time:.1f} seconds"
    )

    print(
        "  Best CV Macro F1: "
        f"{rf_search.best_score_:.4f}"
    )

    print(
        "  Best Parameters: "
        f"{rf_search.best_params_}"
    )

    # --------------------------------------------------------
    # 4B. XGBoost
    # --------------------------------------------------------

    if XGB_AVAILABLE:

        print(
            "\n[B] Tuning XGBoost..."
        )

        xgb_param_dist = {

            "n_estimators": [
                100,
                150,
                200,
            ],

            "max_depth": [
                3,
                5,
                7,
            ],

            "learning_rate": [
                0.03,
                0.08,
                0.15,
            ],

            "subsample": [
                0.8,
                0.9,
                1.0,
            ],

            "colsample_bytree": [
                0.7,
                0.8,
                1.0,
            ],
        }

        xgb_base = XGBClassifier(
            random_state=RANDOM_STATE,
            eval_metric="mlogloss",
            n_jobs=-1,
        )

        xgb_search = RandomizedSearchCV(
            estimator=xgb_base,
            param_distributions=xgb_param_dist,
            n_iter=10,
            cv=cv,
            scoring="f1_macro",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=1,
            refit=True,
        )

        start_time = time.time()

        xgb_search.fit(
            X_train_final,
            y_train,
        )

        xgb_time = time.time() - start_time

        tuned_searches[
            "XGBoost"
        ] = xgb_search

        print(
            f"  Completed in {xgb_time:.1f} seconds"
        )

        print(
            "  Best CV Macro F1: "
            f"{xgb_search.best_score_:.4f}"
        )

        print(
            "  Best Parameters: "
            f"{xgb_search.best_params_}"
        )

    else:

        print(
            "\n[B] XGBoost is not installed."
        )

        print(
            "    XGBoost tuning will be skipped."
        )

    # ========================================================
    # 5. BASELINE VS TUNED COMPARISON
    # ========================================================

    print(
        "\n[5/7] Comparing Stage 6 Baselines "
        "with Stage 7 Tuned Models..."
    )

    comparison_records = []

    for model_name, search in tuned_searches.items():

        baseline = get_baseline(
            model_name
        )

        tuned_cv_f1 = search.best_score_

        baseline_cv_f1 = (
            baseline["CV F1 (Macro)"]
            if baseline is not None
            else np.nan
        )

        baseline_test_f1 = (
            baseline["Test F1 (Macro)"]
            if baseline is not None
            else np.nan
        )

        cv_gain = (
            tuned_cv_f1 - baseline_cv_f1
            if not np.isnan(baseline_cv_f1)
            else np.nan
        )

        comparison_records.append(
            {
                "Model": model_name,

                "Baseline CV F1":
                    baseline_cv_f1,

                "Tuned CV F1":
                    tuned_cv_f1,

                "CV F1 Improvement":
                    cv_gain,

                "Baseline Test F1":
                    baseline_test_f1,

                "Tuned Best CV Rank":
                    tuned_cv_f1,
            }
        )

    comparison_df = pd.DataFrame(
        comparison_records
    )

    comparison_df = comparison_df.sort_values(
        "Tuned CV F1",
        ascending=False,
    ).reset_index(drop=True)

    print(
        "\nBaseline vs Tuned Comparison:"
    )

    print(
        comparison_df.to_string(
            index=False,
            float_format=lambda value: f"{value:.4f}",
        )
    )

    comparison_df.to_csv(
        "stage7_baseline_vs_tuned_results.csv",
        index=False,
    )

    # ========================================================
    # 6. FINAL MODEL SELECTION
    # ========================================================

    print(
        "\n[6/7] Final Model Selection..."
    )

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Final model is selected ONLY using CV Macro F1.
    #
    # The test set is NOT used to select the model.
    # --------------------------------------------------------

    final_candidates = []

    for model_name, search in tuned_searches.items():

        final_candidates.append(
            {
                "Model": model_name,
                "CV Macro F1": search.best_score_,
                "Best Parameters": str(
                    search.best_params_
                ),
            }
        )

    final_candidates_df = pd.DataFrame(
        final_candidates
    ).sort_values(
        "CV Macro F1",
        ascending=False,
    ).reset_index(drop=True)

    print(
        "\nTuned Model Ranking Based on CV Macro F1:"
    )

    print(
        final_candidates_df.to_string(
            index=False,
            float_format=lambda value: f"{value:.4f}",
        )
    )

    # --------------------------------------------------------
    # Select model using CV only
    # --------------------------------------------------------

    final_model_name = (
        final_candidates_df.iloc[0]["Model"]
    )

    final_search = tuned_searches[
        final_model_name
    ]

    final_model = (
        final_search.best_estimator_
    )

    final_cv_f1 = (
        final_search.best_score_
    )

    print(
        "\n>>> FINAL MODEL SELECTED USING "
        "CROSS-VALIDATED MACRO F1 <<<"
    )

    print(
        f"Model: {final_model_name}"
    )

    print(
        f"Best CV Macro F1: {final_cv_f1:.4f}"
    )

    print(
        f"Best Parameters: "
        f"{final_search.best_params_}"
    )

    # ========================================================
    # 7. FINAL TEST EVALUATION
    # ========================================================

    print(
        "\n[7/7] Final Evaluation on Hold-Out Test Set..."
    )

    # --------------------------------------------------------
    # The test set is used only AFTER model selection.
    # --------------------------------------------------------

    final_pred = final_model.predict(
        X_test_final
    )

    final_test_accuracy = accuracy_score(
        y_test,
        final_pred,
    )

    final_test_precision = precision_score(
        y_test,
        final_pred,
        average="macro",
        zero_division=0,
    )

    final_test_recall = recall_score(
        y_test,
        final_pred,
        average="macro",
        zero_division=0,
    )

    final_test_f1 = f1_score(
        y_test,
        final_pred,
        average="macro",
        zero_division=0,
    )

    print(
        "\nFINAL MODEL PERFORMANCE"
    )

    print(
        f"  Model              : "
        f"{final_model_name}"
    )

    print(
        f"  CV Macro F1        : "
        f"{final_cv_f1:.4f}"
    )

    print(
        f"  Test Accuracy      : "
        f"{final_test_accuracy:.4f}"
    )

    print(
        f"  Test Precision     : "
        f"{final_test_precision:.4f}"
    )

    print(
        f"  Test Recall        : "
        f"{final_test_recall:.4f}"
    )

    print(
        f"  Test Macro F1      : "
        f"{final_test_f1:.4f}"
    )

    # --------------------------------------------------------
    # Classification report
    # --------------------------------------------------------

    print(
        "\nDetailed Classification Report:"
    )

    print(
        classification_report(
            y_test,
            final_pred,
            target_names=class_names,
            digits=4,
            zero_division=0,
        )
    )

    # --------------------------------------------------------
    # Confusion matrix
    # --------------------------------------------------------

    print(
        "Confusion Matrix:"
    )

    cm = confusion_matrix(
        y_test,
        final_pred,
    )

    cm_df = pd.DataFrame(
        cm,
        index=[
            f"Actual {class_name}"
            for class_name in class_names
        ],
        columns=[
            f"Predicted {class_name}"
            for class_name in class_names
        ],
    )

    print(
        cm_df.to_string()
    )

    # --------------------------------------------------------
    # Save final test evaluation
    # --------------------------------------------------------

    final_test_results = pd.DataFrame(
        [
            {
                "Final Model":
                    final_model_name,

                "CV Macro F1":
                    final_cv_f1,

                "Test Accuracy":
                    final_test_accuracy,

                "Test Precision (Macro)":
                    final_test_precision,

                "Test Recall (Macro)":
                    final_test_recall,

                "Test F1 (Macro)":
                    final_test_f1,
            }
        ]
    )

    final_test_results.to_csv(
        "stage7_final_model_test_results.csv",
        index=False,
    )

    # ========================================================
    # SAVE ARTIFACTS
    # ========================================================

    print(
        "\nSaving final artifacts..."
    )

    joblib.dump(
        final_model,
        "final_optimized_model.joblib",
    )

    joblib.dump(
        label_encoder,
        "label_encoder.joblib",
    )

    final_candidates_df.to_csv(
        "stage7_model_selection_results.csv",
        index=False,
    )

    print(
        "  Saved: final_optimized_model.joblib"
    )

    print(
        "  Saved: label_encoder.joblib"
    )

    print(
        "  Saved: stage7_model_selection_results.csv"
    )

    print(
        "  Saved: stage7_final_model_test_results.csv"
    )

    print(
        "  Saved: stage7_feature_selection_results.csv"
    )

    print(
        "  Saved: stage7_baseline_vs_tuned_results.csv"
    )

    # ========================================================
    # STAGE 8 / VIVA TECHNICAL JUSTIFICATION
    # ========================================================

    print("\n" + "=" * 80)
    print(
        "STAGE 8 / VIVA TECHNICAL JUSTIFICATION"
    )
    print("=" * 80)

    print(
        "\n1. Why RandomizedSearchCV?"
    )

    print(
        "   RandomizedSearchCV was used to efficiently explore "
        "different hyperparameter combinations without evaluating "
        "every possible combination in the search space."
    )

    print(
        "\n2. Why Stratified 5-Fold Cross-Validation?"
    )

    print(
        "   Stratification preserves the class distribution across "
        "folds, which is important for this imbalanced multiclass "
        "classification problem."
    )

    print(
        "\n3. Why Macro F1?"
    )

    print(
        "   Macro F1 gives equal importance to every class and "
        "balances precision and recall. This prevents the majority "
        "class from dominating the evaluation."
    )

    print(
        "\n4. Feature Selection Investigation:"
    )

    print(
        f"   All features CV Macro F1: "
        f"{all_features_cv_f1:.4f}"
    )

    print(
        f"   Top 100 features CV Macro F1: "
        f"{top100_cv_f1:.4f}"
    )

    print(
        f"   Selected strategy based on CV: "
        f"{selected_feature_strategy}"
    )

    print(
        "\n5. Baseline vs Tuned Models:"
    )

    print(
        "   Hyperparameter tuning was compared against the "
        "Stage 6 baseline using cross-validated Macro F1."
    )

    print(
        "\n6. Final Model Selection:"
    )

    print(
        f"   The final model was selected based on the highest "
        f"cross-validated Macro F1 ({final_cv_f1:.4f})."
    )

    print(
        "   The hold-out test set was not used to choose the model."
    )

    print(
        "\n7. Final Test Evaluation:"
    )

    print(
        f"   After model selection, the final model was evaluated "
        f"on the hold-out test set."
    )

    print(
        f"   Final Test Macro F1: "
        f"{final_test_f1:.4f}"
    )

    print(
        "\n8. Interpretation:"
    )

    print(
        "   Final conclusions should be based on the observed "
        "CV and test results, class-wise precision/recall, and "
        "the confusion matrix rather than assuming a particular "
        "algorithm will always perform best."
    )

    print("\n" + "=" * 80)
    print("STAGE 7 COMPLETED")
    print("=" * 80)

## 4. Run Stage 7

Executes the full optimization pipeline.

> **Note:** `RandomizedSearchCV` with 5-fold CV runs 50 model fits per algorithm, plus the feature-selection comparison. Expect this cell to take a while.

Files produced in the working directory:
- `final_optimized_model.joblib`
- `label_encoder.joblib`
- `stage7_feature_selection_results.csv`
- `stage7_baseline_vs_tuned_results.csv`
- `stage7_model_selection_results.csv`
- `stage7_final_model_test_results.csv`

In [ ]:
if __name__ == "__main__":
    main()

## Wrap-up

- Base conclusions on the CV and test results, the class-wise precision and recall, and the confusion matrix.
- Do not assume that any one algorithm always wins.
- The saved model and label encoder can now be loaded with `joblib.load(...)` for inference or deployment.